# Training a MACE model


In this notebook, the flow of training a MACE model is detailed step by step.



In [ ]:
import matplotlib.pyplot    as plt
plt.ioff()
import numpy                as np
import torch
from time                   import time
import datetime             as dt
from tqdm                   import tqdm

from pathlib import Path
import sys                                                          

parentpath = str(Path().cwd())+'/'

print(parentpath)

sys.path.append(parentpath)

import src.mace.CSE_0D.dataset  as ds
import src.mace.CSE_0D.plotting as plotting
import src.mace.train           as train
import src.mace.test            as test
import src.mace.load            as load
import src.mace.mace            as mace
from src.mace.loss              import Loss
import src.mace.loss            as loss
import src.mace.utils           as utils
from src.mace.meta              import Meta

specs_dict = utils.get_specs()

%reload_ext autoreload
%autoreload 2

## Setting up

>***Note***:
>If you are training the architecture with your own data, you can set up the model in the following way. However, if you will follow the notebook, we will load a pre-trained model to continue the training, just for tutorial purposes. Then, skip the cells on "Set up the model".


### Naming the model
The default model to name the model is with the date and time of training. This can be adjusted with changing the parameter name.




In [ ]:
now = dt.datetime.now()
name = str(now.strftime("%Y%m%d")+'_'+now.strftime("%H%M%S"))
name = "example_out_v2/" + name

path = parentpath + 'model/'+name

print(path)

Make directories to store output and trained model.



In [ ]:
utils.makeOutputDir(path)
utils.makeOutputDir(path+'/nn')
utils.makeOutputDir(path+'/figs')

### Reading input file
Fill in the name of the input file for arg.

For tutorial purposes, the amount of training data and training epochs is limited in this example.

In [ ]:
arg = 'example_v2'

Meta files can be intialised from a path/meta.json or a given input.in file.

In [ ]:
infile = parentpath+'input/'+arg+'.in'
meta = Meta(infile=infile, name=name)
meta.print()

### Set up Pytorch

In [ ]:
cuda   = False
DEVICE = torch.device("cuda" if cuda else "cpu")
batch_size = 1
kwargs = {'num_workers': 1, 'pin_memory': True}

### Set up the model


Make the model, randomly initialised.



In [ ]:
model = mace.MACE(meta.n_dim, meta.p_dim, meta.z_dim,
                  ae_type=meta.ae_type, g_nn=meta.g_nn, 
                  atol=meta.atol, rtol=meta.rtol)

num_params = utils.count_parameters(model)
print(f'\nThe model has {num_params} trainable parameters')

### Load training & validation data

Load the train, validation, and test data sets.

In [ ]:
traindata, testdata, data_loader, test_loader = ds.get_data(dt_fract=meta.dt_fract,
                                                            nb_samples=meta.nb_train, batch_size=batch_size,
                                                            nb_test=meta.nb_valid,kwargs=kwargs, inpackage=True, data_type=meta.data_type)

## Loading the pre-trained model


Setting the directory of the model.



In [ ]:
outloc  = parentpath+'model/'
dirname = 'example_weights_v2'
epoch   = 7

You can load a MACE model from a directory to get a solver, loss, and meta objects.

In [ ]:
model, losses, metadata = load.load_MACE(f'model/{dirname}/', epoch=epoch)
print(f'The model has {utils.count_parameters(model)} trainable parameters')

Alternatively, you can overwrite the weights of an initialised model.

In [ ]:
load.overwrite_weights(model, f'model/{dirname}/', epoch=epoch)

## Training the model

Training the model in [Maes et al. (2024)](https://ui.adsabs.harvard.edu/abs/2024arXiv240503274M/abstract) happens it two stages:

1. For the first few epochs (meta.ini_epochs) the models is trained with unscaled losses.
2. The losses of the model are rescaled, normalised based in the losses in the first few epochs.

### Stage 1
Initialise loss and exit code.

In [ ]:
trainloss = loss.setup_loss(meta.lossweights)
validloss = loss.setup_loss(meta.lossweights)
exit_code = 0 # nODE solver status, optional

Train the model & time the training.

In [ ]:
lr = meta.lr
lr = 1e-4
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [ ]:
exit_code +=train.train(model,
            data_loader, test_loader,
            trainloss, validloss,
            optimiser=optimizer, 
            nb_epochs=meta.ini_epochs,
            nb_evol=meta.nb_evol,
            batch_size=8)

In [ ]:
loss.plot({'train':trainloss, 'valid':validloss})

### Stage 2
Normalise losses.

In [ ]:
trainloss.normalize()
validloss.normalize()

In [ ]:
lr = meta.lr
lr = 1e-3 # violence
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

Continue the training. Use lower batch count initially.


In [ ]:
exit_code +=train.train(model,
            data_loader, test_loader,
            trainloss, validloss,
            optimiser=optimizer, 
            nb_epochs=30,
            nb_evol=meta.nb_evol,
            batch_size=8)

### Stage 3

Normalize losses again.

In [ ]:
trainloss.normalize()
validloss.normalize()

In [ ]:
lr = meta.lr
lr = 1e-4
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

Continue training with full batches.

In [ ]:
exit_code +=train.train(model,
            data_loader, test_loader,
            trainloss, validloss,
            optimiser=optimizer, 
            nb_epochs=50,
            nb_evol=meta.nb_evol)

In [ ]:
loss.plot({'train':trainloss, 'valid':validloss})

## Saving the model
Save the losses & specifics of the data set.

In [ ]:
import os 
print('path:', path)

trainloss.save(path+'/train')
validloss.save(path+'/valid')

Save the model and the status of the solver (see [torchode status](https://github.com/martenlienen/torchode/blob/main/torchode/status_codes.py) for more details).

In [ ]:
torch.save(model.state_dict(),path+'/nn/nn.pt')
meta.updateMeta(path=path, exit_code=exit_code)

Plot the evolution of the loss and save the figure.



In [ ]:
plt.close()
loss.plot({'train':trainloss, 'valid':validloss})
plt.savefig(path+'/figs/loss.png')
plt.show()

## Testing the model


Perform tests on the trained model to get an accuracy indication, according to the following error metric:

$$ {\rm error} = \frac{\log_{10}\boldsymbol{n}-\log_{10}\hat{\boldsymbol{n}}}{\log_{10}\boldsymbol{n}}, $$

which is executed element-wise and subsequently summed over the different chemical species

### Performing the tests

In [ ]:
errors, times = test.testModel(model, test_loader)
print(errors)
print(times)

### Saving the outcome of the tests

The error and times are averaged over the amount of tests.

In [ ]:
meta.updateMeta(test_errors=errors, test_times=times)

### Plotting abundances

In [ ]:
testpath = 'mid_dens' # TODO: fix ds.get_test_data
model1D, (n, p, dt), info = ds.get_test_data('1DCSE', testpath, meta, inpackage=True)

In [ ]:
tic = time()
with torch.no_grad():
    n_hat, status = model.rollout(n[0:1], p[None], dt[None])
toc = time()

n_evol = torch.cat((n[0:1], n_hat[0]), dim=0)      ## (N, 468)

print(f'Evolution of {n_hat.shape[1]} steps in {toc-tic:.2f} s')
print('Solver status counts [success, general error, dt_min, max_steps, inf. norm]:',
      torch.bincount(status.flatten(), minlength=5).tolist())

In [ ]:
n_real = ds.get_abs(n.numpy())
n_evol = ds.get_abs(n_evol.numpy())

err, err_evol = utils.error(n_real, n_evol)

print('Evolution error (Eq. 23):', np.round(err_evol, 3))

In [ ]:
fig12_specs = ['CO', 'H2O', 'OH', 'C2H2', 'C2H', 'CH3C5NH+', 'C10H2+']

fig, axes = plotting.plot_abs(model1D, '1DCSE', n_real, n_evol, specs=fig12_specs, step=False)

#axes[2].set_ylim(-.5,.1)
fig.suptitle(f'MACE v2, {testpath}', y=1.0)
fig.savefig(path+'/figs/preds.png')
plt.show()